In [ ]:
# %% [markdown]
# # 📂 Generador de Expediente de Evidencias Críticas
# Crea un careo directo entre las líneas con más del 50% de similitud.

# %%
import requests, re, datetime
from pathlib import Path
from tqdm import tqdm
import tkinter as tk
from tkinter import filedialog

# --- CONFIGURACIÓN ---
MODELO_OLLAMA = "gemma3n:e4b" 
URL_OLLAMA = "http://localhost:11434/api/generate"


In [ ]:
# %%
# --- 1. CARGA DE ARCHIVOS ---
root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
path_a = filedialog.askopenfilename(title="Tesis A (Referencia)", filetypes=[("Markdown", "*.md")])
path_b = filedialog.askopenfilename(title="Tesis B (Sospecha)", filetypes=[("Markdown", "*.md")])
root.destroy()

def leer_lineas_dict(ruta):
    with open(ruta, "r", encoding="utf-8") as f:
        # Guardamos en un diccionario {numero_linea: texto}
        return {i+1: l.strip() for i, l in enumerate(f.readlines()) if len(l.strip()) > 20}

dict_a = leer_lineas_dict(path_a)
dict_b = leer_lineas_dict(path_b)


In [ ]:
# %%
# --- 2. PROCESAMIENTO DE EVIDENCIAS CRÍTICAS ---
evidencias = []
contador = 0

print(f"🔎 Buscando evidencias críticas en {len(dict_b)} líneas...")

# Ordenamos las líneas para que el reporte sea secuencial
for n_b, txt_b in tqdm(sorted(dict_b.items()), desc="Comparando"):
    words_b = set(re.findall(r'\w+', txt_b.lower()))
    
    for n_a, txt_a in dict_a.items():
        words_a = set(re.findall(r'\w+', txt_a.lower()))
        
        # Cálculo matemático de similitud
        comunes = words_a.intersection(words_b)
        union = words_a.union(words_b)
        sim = (len(comunes) / len(union)) * 100 if union else 0
        
        # FILTRO: Solo Críticos y Sospechosos (> 50%)
        if sim >= 50:
            contador += 1
            
            # Opcional: Validar con IA solo para dar un dictamen cualitativo
            # Si prefieres velocidad, puedes comentar las siguientes 2 líneas
            prompt = f"¿Es el texto B una copia/paráfrasis del A? Responde en 5 palabras.\nA: {txt_a}\nB: {txt_b}"
            try:
                res = requests.post(URL_OLLAMA, json={"model": MODELO_OLLAMA, "prompt": prompt, "stream": False}, timeout=5)
                dictamen_ia = res.json().get('response', 'Sin dictamen').strip()
            except:
                dictamen_ia = "Error de conexión con IA"

            # Construir el bloque de evidencia
            bloque = (
                f"### Hallazgo {contador}: Similitud {sim:.2f}% [CRÍTICO]\n"
                f"**Ubicación:** Línea #{n_a} (Tesis A) vs Línea #{n_b} (Tesis B)\n"
                f"**Dictamen IA:** {dictamen_ia}\n\n"
                f"**Evidencia:**\n"
                f"- **Tesis A (L#{n_a}):** `{txt_a}`\n"
                f"- **Tesis B (L#{n_b}):** `{txt_b}`\n"
                f"\n---\n"
            )
            evidencias.append(bloque)


In [ ]:
# %%
# --- 3. GENERACIÓN DEL ARCHIVO DE EXPEDIENTE ---
reporte_header = f"""# 🚨 EXPEDIENTE DE EVIDENCIAS CRÍTICAS DE PLAGIO
**Fecha de Auditoría:** {datetime.datetime.now().strftime('%d/%m/%Y %H:%M')}
**Documento A (Referencia):** {Path(path_a).name}
**Documento B (Sospecha):** {Path(path_b).name}

## RESUMEN
Se han encontrado **{len(evidencias)}** coincidencias críticas con un nivel de similitud superior al 75%.

---
"""

path_expediente = Path(path_b).parent / f"EXPEDIENTE_EVIDENCIAS_{Path(path_b).stem}.md"

with open(path_expediente, "w", encoding="utf-8") as f:
    f.write(reporte_header)
    for ev in evidencias:
        f.write(ev)

print(f"\n✨ Expediente generado con {len(evidencias)} hallazgos.")
print(f"📂 Archivo: {path_expediente.name}")